# Evaluación 2 · Sección 5 — Hallazgos y siguientes pasos

**Alcance de este notebook:** Documentar hallazgos

## Resumen numérico de lo encontrado en las Secciones 1 a 4

In [1]:
import pandas as pd

resumen = pd.DataFrame({
    "Métrica": [
        "Postulantes que quedaron fuera de todas sus preferencias",
        "De ellos, con alternativa alcanzable en su área de interés",
        "De ellos, sin alternativa alcanzable",
        "% con alternativa alcanzable — Particular Pagado",
        "% con alternativa alcanzable — resto de dependencias",
        "Mejor puntaje promedio — grupo SIN alternativa",
        "Mejor puntaje promedio — grupo CON alternativa",
        "Validación independiente del puntaje ponderado (error < 0.5 pts)",
    ],
    "Valor": [
        "27.492",
        "20.336 (73,97%)",
        "7.156 (26,03%)",
        "56,8%",
        "73,7% – 79,8%",
        "713,4",
        "619,7",
        "99,96% (472 discrepantes, 86,2% con causa identificada)",
    ],
})
resumen

,Métrica,Valor
0,Postulantes que quedaron fuera de todas sus pr...,27.492
1,"De ellos, con alternativa alcanzable en su áre...","20.336 (73,97%)"
2,"De ellos, sin alternativa alcanzable","7.156 (26,03%)"
3,% con alternativa alcanzable — Particular Pagado,"56,8%"
4,% con alternativa alcanzable — resto de depend...,"73,7% – 79,8%"
5,Mejor puntaje promedio — grupo SIN alternativa,"713,4"
6,Mejor puntaje promedio — grupo CON alternativa,"619,7"
7,Validación independiente del puntaje ponderado...,"99,96% (472 discrepantes, 86,2% con causa iden..."


## Hallazgos

**1. Casi 3 de cada 4 postulantes que quedaron fuera sí tenían dónde ir.** El 73,97% de los 27.492 postulantes que no quedaron seleccionados en ninguna de sus preferencias contaba con un puntaje ponderado suficiente para haber sido seleccionado en al menos un programa de su propia área de interés con vacantes no cubiertas. Esto confirma, con un número concreto y verificable, la magnitud del problema que motivó el proyecto: la mayoría de quienes quedan fuera no lo hacen por falta de puntaje, sino por un desencuentro entre su lista de preferencias y la oferta real disponible.

**2. La hipótesis original de la formulación no se sostiene — y el resultado corre en la dirección contraria.** Contrario a lo planteado inicialmente, los postulantes de establecimientos particulares pagados muestran la *menor* proporción de alternativa alcanzable (56,8%) entre las seis dependencias, frente a un rango de 73,7% a 79,8% en el resto. La causa más plausible no es un puntaje insuficiente —de hecho, es el grupo con mayor puntaje promedio del sistema (Sección 3)—, sino que sus áreas de interés concentran carreras con un margen de sustitución nacional más angosto: en promedio, evalúan 12,8 programas alternativos dentro de su interés, frente a 18-24 en las demás dependencias.

**3. El grupo sin alternativa alcanzable es de alto puntaje, no de bajo rendimiento.** Entre quienes no lograron ninguna alternativa, el mejor puntaje ponderado promedio (713,4) es más alto que el de quienes sí la lograron (619,7). El problema no es de capacidad académica: es de exceso de selectividad en los programas elegidos, sin un margen de respaldo suficiente en la propia lista.

**4. Los ceros en las cinco pruebas PAES son faltantes disfrazados, no datos faltantes al azar.** Para `CLEC_MAX` y `MATE1_MAX`, el contraste contra la variable de registro actual mostró que el cero se comporta como "no rindió" y no como desempeño: la variable `_MAX` rescata un 5,7% de estudiantes que el registro del proceso vigente pierde, porque rindieron la prueba en otro proceso o fecha. Para `MATE2_MAX`, `HCSOC_MAX` y `CIEN_MAX`, el cero se explica por su carácter de prueba optativa, confirmado por el código `35` de `ESTADO_PREF` ("no rindió ninguna de las pruebas opcionales"). En ningún caso el cero representa un desempeño real de cero puntos, que además sería imposible en una escala que parte en 100.

**5. Resolver valores disfrazados no documentados fue condición necesaria para no sesgar el resultado central.** El código 99 de `INGRESO_PERCAPITA_GRUPO_FA`, el espacio en blanco de `PTJE_PREF`, y la redistribución de ponderaciones para postulantes extranjeros sin NEM (documentada oficialmente por DEMRE) no estaban resueltos en la formulación original. De haberse tratado como ceros en vez de reconocerlos y corregirlos, el cálculo de alternativa alcanzable habría subestimado el puntaje de un grupo que no está distribuido al azar entre dependencias. No cuantificamos ese sesgo —habría que recalcular todo con el tratamiento incorrecto para medirlo—, pero basta para justificar por qué resolver estos tres casos era condición previa al resultado central.

## Qué modelo imaginamos ajustar

Siguiendo el criterio de partir simple y avanzar (visto en clase): como primer modelo, muy básico, ajustaríamos una regresión lineal (OLS) de `tiene_alternativa_alcanzable` en función del puntaje ponderado y el decil de ingreso — aun sabiendo que no es la herramienta ideal para una variable binaria, sirve como punto de partida rápido de interpretar. Como segundo modelo, más adecuado, imaginamos una regresión logística con las mismas variables más `DEPENDENCIA` (convertida a variables dummy con `pd.get_dummies`), comparando el ajuste con y sin las variables categóricas, y contrastando el resultado con y sin los atípicos ya identificados en la Sección 2 — dado que, como vimos en el contraste de la Sección 3, no siempre conviene eliminarlos sin medir antes el efecto.

Esta es una intención inicial de una fase temprana del proyecto (el diplomado dura 6 meses); es muy probable que cambie a medida que se profundice, en particular si se logra refinar la definición de "área de interés" más allá de la carrera exacta declarada por cada estudiante.

## Uso de Inteligencia Artificial

Se utilizó **Claude (Anthropic)** como asistente para la redacción de los hallazgos de este notebook, la verificación de que cada cifra citada corresponda a la salida de los notebooks anteriores, y la revisión de las afirmaciones que excedían lo que los datos muestran. La selección de qué constituye un hallazgo, su interpretación y la propuesta de modelo para la etapa siguiente son del equipo; todas las cifras son reproducibles sobre las bases públicas declaradas.